---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: AI Engineering

### 📋 **Homework 5**: RAG System for Fordham University

### 📅 **Due Date**: Day of Lecture 7, 11:59 PM

### Difficulty: ★★★★☆


**Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.

---

In this homework, you'll complete the RAG (Retrieval Augmented Generation) system you started building in Lecture 5. You will build an end-to-end pipeline that can answer questions about Fordham University using real data scraped from the Fordham website.

This is an open-ended assignment — there is no single right implementation, and you're encouraged to experiment with chunking strategies, embedding models, prompt design, and retrieval parameters to improve your system. **I will grade your system by testing it with specific questions that I know the answer to.**

---

## Instructions

- You may use ChatGPT, Claude, documentation, Stack Overflow, etc. When using external resources, briefly cite them in a comment.
- Your submission must include **pre-computed embeddings** and any other artifacts needed so that I can run your RAG system without recomputing anything expensive. Share it with us in a way that makes sense.
- Run all cells before submitting to ensure they work.

**Submission:**
1. Create a branch called `homework-5`
2. Commit and push your work (notebook + Streamlit app + saved embeddings/artifacts in `temp/`)
3. Create a PR and merge to main
4. Submit the `.ipynb` file on Blackboard

---

## Step 1: Load and Chunk the Fordham Website Data

In `data/fordham-website.zip` you'll find **~9,500 Markdown files** scraped from Fordham's website. Each file is one page — admissions info, program descriptions, faculty pages, financial aid, campus life, and more. The first line of every file is the **URL** of the page it was scraped from. The rest is the page content in Markdown.

Think about: chunk size, what to split on (paragraphs, headers, fixed length, etc.), whether chunks should overlap, and how to track which page each chunk came from.

In [1]:
import os
import pandas as pd
from pathlib import Path

# Path to your data folder (adjust this to match your setup)
DATA_DIR = 'data'  # or 'data/fordham-website' - whatever folder has the .md files

# Load all markdown files
documents = []

for md_file in Path(DATA_DIR).rglob('*.md'):  # rglob finds files in subfolders too
    with open(md_file, 'r', encoding='utf-8') as f:
        content = f.read()
        
    # Split into lines
    lines = content.split('\n')
    
    # First line is URL, rest is content
    url = lines[0].strip() if lines else ""
    page_content = '\n'.join(lines[1:]).strip()
    
    if page_content:  # Skip empty files
        documents.append({
            'filename': md_file.name,
            'url': url,
            'content': page_content,
            'length': len(page_content)
        })

print(f"Loaded {len(documents)} documents")

# Show some stats
df = pd.DataFrame(documents)
print(f"\nContent length stats:")
print(df['length'].describe())

Loaded 9530 documents

Content length stats:
count      9530.000000
mean       4184.807660
std        9023.852434
min         103.000000
25%        1268.250000
50%        2270.000000
75%        4370.750000
max      431113.000000
Name: length, dtype: float64


In [2]:
# Simple chunking - no fancy libraries needed
all_chunks = []
chunk_id = 0

print("Creating chunks...")
for i, doc in enumerate(documents):
    if i % 1000 == 0:  # Progress update every 1000 docs
        print(f"Processing document {i}/{len(documents)}...")
    
    content = doc['content']
    
    # Simple: if document is short, keep it as one chunk
    if len(content) <= 1000:
        all_chunks.append({
            'chunk_id': chunk_id,
            'doc_filename': doc['filename'],
            'doc_url': doc['url'],
            'text': content
        })
        chunk_id += 1
    else:
        # Split long documents every 1000 characters at word boundaries
        start = 0
        while start < len(content):
            end = start + 1000
            if end < len(content):
                # Find last space before 1000 chars
                last_space = content[start:end].rfind(' ')
                if last_space > 0:
                    end = start + last_space
            
            chunk = content[start:end].strip()
            if chunk:
                all_chunks.append({
                    'chunk_id': chunk_id,
                    'doc_filename': doc['filename'],
                    'doc_url': doc['url'],
                    'text': chunk
                })
                chunk_id += 1
            
            start = end

print(f"\nDone! Created {len(all_chunks)} chunks")

Creating chunks...
Processing document 0/9530...
Processing document 1000/9530...
Processing document 2000/9530...
Processing document 3000/9530...
Processing document 4000/9530...
Processing document 5000/9530...
Processing document 6000/9530...
Processing document 7000/9530...
Processing document 8000/9530...
Processing document 9000/9530...

Done! Created 45020 chunks


In [3]:
# Check the chunks
print("Chunk statistics:")
chunk_lengths = [len(c['text']) for c in all_chunks]
print(f"Total chunks: {len(all_chunks)}")
print(f"Average chunk length: {sum(chunk_lengths) / len(chunk_lengths):.1f} characters")
print(f"Min: {min(chunk_lengths)}, Max: {max(chunk_lengths)}")

# Look at a few examples
print("\n" + "="*80)
print("Example chunks:")
print("="*80)
for i in range(3):
    chunk = all_chunks[i]
    print(f"\nChunk {i+1}:")
    print(f"From: {chunk['doc_url']}")
    print(f"Length: {len(chunk['text'])} chars")
    print(f"Text preview: {chunk['text'][:300]}...")
    print("-"*80)

Chunk statistics:
Total chunks: 45020
Average chunk length: 885.1 characters
Min: 2, Max: 1000

Example chunks:

Chunk 1:
From: https://www.fordham.edu/about/living-the-mission/campus-ministry/catholic-life/ministry-of-music
Length: 996 chars
Text preview: # Ministry of Music


Fordham offers each student a wealth of musical opportunities. Not only can students hear great music in the concert halls and churches of Manhattan but also experience this music firsthand through participation in a university choir. Whether concert oriented, or for the worshi...
--------------------------------------------------------------------------------

Chunk 2:
From: https://www.fordham.edu/about/living-the-mission/campus-ministry/catholic-life/ministry-of-music
Length: 992 chars
Text preview: Lessons and Carols (with the University Choir), the Holy Week Liturgies, and the Baccalaureate Mass. Members of the choir often serve as cantors and leaders of song. Membership is by audition. Undergraduate studen

---

## Step 2: Embed the Chunks

Turn each chunk into a vector so you can search over them. You can use a local model or an API model — your choice.

Once you've created your embeddings, **save them somewhere** so you (and I) don't have to redo this step. Save the chunk metadata too (text, source URL, etc.).

In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np

print("Loading embedding model...")
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print("Model loaded!")

# Extract just the text from chunks
chunk_texts = [chunk['text'] for chunk in all_chunks]

print(f"\nCreating embeddings for {len(chunk_texts)} chunks...")
print("This will take a few minutes...")

# Create embeddings in batches to avoid memory issues
embeddings = model.encode(
    chunk_texts, 
    show_progress_bar=True,
    batch_size=32
)

print(f"\nDone! Created embeddings with shape: {embeddings.shape}")

# Save embeddings so we don't have to recreate them
np.save('embeddings.npy', embeddings)
print("Saved embeddings to embeddings.npy")

/Users/suhaniwadhwa/ai-engineering-fordham/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2848.70it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded!

Creating embeddings for 45020 chunks...
This will take a few minutes...


Batches: 100%|██████████| 1407/1407 [04:19<00:00,  5.43it/s]



Done! Created embeddings with shape: (45020, 384)
Saved embeddings to embeddings.npy


---

## Step 3: Retrieve

Build the **R** in RAG. Write a function that takes a question and returns the most relevant chunks. You can use semantic search, BM25, hybrid — whatever you think works best.

Test it on a few questions and eyeball whether the results make sense.

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_relevant_chunks(question, top_k=5):
    """
    Find the most relevant chunks for a question.
    
    Args:
        question: The user's question
        top_k: Number of chunks to retrieve
    
    Returns:
        List of relevant chunks with their similarity scores
    """
    # Embed the question using the same model
    question_embedding = model.encode([question])
    
    # Compute similarity with all chunks
    similarities = cosine_similarity(question_embedding, embeddings)[0]
    
    # Get top k indices
    top_indices = np.argsort(similarities)[-top_k:][::-1]
    
    # Return chunks with scores
    results = []
    for idx in top_indices:
        results.append({
            'chunk': all_chunks[idx],
            'similarity': float(similarities[idx])
        })
    
    return results

print("Retrieval function ready!")



Retrieval function ready!


---

## Step 4: Generate

Build the **G** in RAG. Write a function that takes a question and the retrieved chunks, builds a prompt, and calls an LLM to generate an answer.

Think about: how to structure the prompt, what the LLM should do when the context doesn't contain the answer, and which model to use.

In [6]:
from openai import OpenAI
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

def generate_answer(question, relevant_chunks, history=None):
    """Generate answer using retrieved chunks AND conversation history."""
    
    # Build context from chunks
    context_parts = []
    for i, result in enumerate(relevant_chunks, 1):
        chunk = result['chunk']
        context_parts.append(f"[Source {i}: {chunk['doc_url']}]\n{chunk['text']}")
    context = "\n\n---\n\n".join(context_parts)
    
    system_prompt = """You are a helpful assistant answering questions about Fordham University. 
Use only the provided context to answer questions. 
If the context doesn't contain enough information, say so honestly.
When answering follow-up questions, use the conversation history to understand what the user is referring to."""
    
    # Build messages list: start with system, add history, add current question
    messages = [{"role": "system", "content": system_prompt}]
    
    # Add conversation history (previous questions + answers)
    if history:
        for turn in history:
            messages.append({"role": "user", "content": turn["question"]})
            messages.append({"role": "assistant", "content": turn["answer"]})
    
    # Add current question with context
    user_prompt = f"""CONTEXT:
{context}

QUESTION: {question}

Please provide a helpful, accurate answer based on the context above. Mention which sources you used."""
    
    messages.append({"role": "user", "content": user_prompt})
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.7,
        max_tokens=500
    )
    
    return response.choices[0].message.content

---

## Step 5: Wire it Together

Combine the previous steps into a single `rag(question)` function. Question in, answer out.

In [9]:
conversation_history = []

def rewrite_query(question, history):
    """Rewrite vague follow-up questions to be self-contained before retrieving."""
    if not history:
        return question
    
    recent = history[-3:]
    history_text = "\n".join([f"User: {t['question']}\nAssistant: {t['answer']}" for t in recent])
    
    prompt = f"""Given this conversation history:
{history_text}

Rewrite this follow-up question to be fully self-contained (no pronouns like "it", "they", "there", "those"):
Follow-up question: {question}

Return ONLY the rewritten question, nothing else."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=100
    )
    return response.choices[0].message.content.strip()


def reset_conversation():
    """Clear conversation history to start fresh."""
    global conversation_history
    conversation_history = []
    print("Conversation history cleared!")


def rag(question, top_k=5, verbose=True, use_history=True):
    """
    Complete RAG pipeline with conversation memory + query rewriting.
    Question in, answer out.
    """
    global conversation_history
    history = conversation_history if use_history else []

    # Rewrite vague follow-ups before retrieving
    search_query = rewrite_query(question, history)

    if verbose:
        if search_query != question:
            print(f"Rewritten query: {search_query}")
        print("Retrieving relevant chunks...")

    # Use rewritten query for retrieval
    relevant_chunks = retrieve_relevant_chunks(search_query, top_k=top_k)

    if verbose:
        print(f"Found {len(relevant_chunks)} chunks")
        for i, result in enumerate(relevant_chunks, 1):
            print(f"  [{i}] {result['chunk']['doc_url'][:70]}... (similarity: {result['similarity']:.3f})")
        print("\nGenerating answer...\n")

    # Generate using ORIGINAL question + history
    answer = generate_answer(question, relevant_chunks, history=history)

    # Save to history
    conversation_history.append({"question": question, "answer": answer})
    if len(conversation_history) > 5:
        conversation_history.pop(0)

    return answer


print("RAG pipeline with conversation memory + query rewriting ready!")

# Test follow-up questions
reset_conversation()
print(rag("What programs does Gabelli School of Business offer?"))
print("---")
print(rag("How much does it cost?"))
print("---")
print(rag("What about financial aid for those programs?"))


RAG pipeline with conversation memory + query rewriting ready!
Conversation history cleared!
Retrieving relevant chunks...
Found 5 chunks
  [1] https://www.fordham.edu/academics/colleges-and-schools/graduate-school... (similarity: 0.774)
  [2] https://www.fordham.edu/gabelli-school-of-business/academic-programs-a... (similarity: 0.761)
  [3] https://www.fordham.edu/gabelli-school-of-business/academic-programs-a... (similarity: 0.749)
  [4] https://www.fordham.edu/gabelli-school-of-business/student-and-career-... (similarity: 0.739)
  [5] https://www.fordham.edu/gabelli-school-of-business/academic-programs-a... (similarity: 0.731)

Generating answer...

The Gabelli School of Business offers a variety of graduate and executive programs designed to advance your career. Specifically, they provide:

- Three types of M.B.A. programs: full-time, professional, and executive M.B.A.
- Twelve M.S. programs, with two offered online.
- Two doctoral programs: Ph.D. and Doctor of Professional Studies

In [10]:
# Demonstrate your RAG system

demo_questions = [
    "What programs does the Gabelli School of Business offer?",
    "How do I apply for financial aid at Fordham?",
    "What is the tuition for undergraduate students?",
    "Tell me about Fordham's campus locations.",
    "What research opportunities are available for students?",
]

reset_conversation()  # start fresh for demo

for q in demo_questions:
    print(f"Q: {q}")
    answer = rag(q, verbose=False)  # verbose=False keeps output clean
    print(f"A: {answer}")
    print("-" * 80)

Conversation history cleared!
Q: What programs does the Gabelli School of Business offer?
A: The Gabelli School of Business offers a variety of graduate and executive programs, including:

1. **M.B.A. Programs**: There are three types of M.B.A. programs available:
   - Full-time M.B.A.
   - Professional M.B.A.
   - Executive M.B.A.

2. **M.S. Programs**: The school offers 12 Master of Science (M.S.) programs, with two of them available online.

3. **Doctoral Programs**: There are two doctoral programs:
   - Ph.D.
   - Doctor of Professional Studies.

Additionally, the Gabelli School emphasizes a dual core curriculum that integrates business and liberal arts education, ensuring students gain a comprehensive grounding in various business disciplines. 

These details are sourced from [Source 1](https://www.fordham.edu/academics/colleges-and-schools/graduate-schools) and [Source 2](https://www.fordham.edu/gabelli-school-of-business/academic-programs-and-admissions).
-----------------------

---

## Step 6: Evaluate Your RAG System

A working RAG system is great — but how do you know it's actually *good*? You can't improve what you can't measure. In this step you'll build an evaluation framework using concepts from Lecture 6.

There are two things to evaluate in a RAG system:
- **Retrieval quality**: Are you finding the right chunks?
- **Answer quality**: Is the generated answer correct and grounded in the context?

### Build a test set

Create a test set of at least **10 question-answer pairs**. For each pair, provide the question and the expected answer (look it up in the data). Cover a range of question types — factual, procedural, about specific programs, etc.

### Evaluate retrieval

For each question, check whether the retrieved chunks actually contain relevant information. You can do this manually or automatically (e.g., use an LLM to judge relevance). Compute **context relevance** — the fraction of retrieved chunks that are actually useful.

### Evaluate answers with LLM-as-judge

Use an LLM to evaluate your system's answers on two dimensions:

1. **Faithfulness**: Does the answer only use information from the retrieved context? (No hallucination)
2. **Correctness**: Is the answer factually correct compared to the expected answer?

Use **structured outputs** (Pydantic) to get consistent scores from the judge. A starting schema is provided below — feel free to modify it.

In [11]:
from pydantic import BaseModel, Field

class RAGEvaluation(BaseModel):
    faithfulness_score: int = Field(
        ..., ge=1, le=5,
        description="1=completely hallucinated, 5=fully grounded in context"
    )
    faithfulness_reasoning: str = Field(
        ..., description="Brief explanation of the faithfulness score"
    )
    correctness_score: int = Field(
        ..., ge=1, le=5,
        description="1=completely wrong, 5=fully correct and complete"
    )
    correctness_reasoning: str = Field(
        ..., description="Brief explanation of the correctness score"
    )

In [12]:
import json

#Test set
test_groups = [
    # Group 1: standalone questions
    [
        {"question": "What is the Yellow Ribbon Program at Fordham?",
         "expected": "A program covering tuition for veterans under the Post-9/11 GI Bill"},
        {"question": "Where is Fordham's Rose Hill campus?",
         "expected": "The Bronx, New York"},
        {"question": "What is the Gabelli School of Business?",
         "expected": "Fordham's business school offering undergraduate and graduate business programs"},
        {"question": "How do I apply for financial aid at Fordham?",
         "expected": "By completing the FAFSA and contacting the Office of Student Financial Services"},
        {"question": "What graduate programs does the School of Education offer?",
         "expected": "Programs in teaching, counseling, and educational leadership"},
        {"question": "What is the FAFSA?",
         "expected": "Free Application for Federal Student Aid"},
        {"question": "How many campuses does Fordham have?",
         "expected": "Three: Rose Hill in the Bronx, Lincoln Center in Manhattan, and Westchester"},
        {"question": "What is Fordham's Jesuit identity?",
         "expected": "Fordham is a Jesuit university founded in 1841"},
    ],
    # Group 2: follow-up questions (DO NOT reset between these)
    [
        {"question": "Does Fordham have a law school?",
         "expected": "Yes, Fordham School of Law"},
        {"question": "What campus is the law school on?",
         "expected": "Lincoln Center campus in Manhattan"},
    ],
]

#Judge function
def evaluate_rag_answer(question, expected_answer, rag_answer, context):
    """Use GPT as a judge to score faithfulness and correctness."""
    
    judge_prompt = f"""You are evaluating a RAG system's answer. Score it on two dimensions.

QUESTION: {question}
EXPECTED ANSWER: {expected_answer}
RAG SYSTEM'S ANSWER: {rag_answer}
CONTEXT PROVIDED TO RAG: {context[:2000]}

Return ONLY a valid JSON object with exactly these keys, no extra text:
{{
  "faithfulness_score": <integer 1-5>,
  "faithfulness_reasoning": "<one sentence>",
  "correctness_score": <integer 1-5>,
  "correctness_reasoning": "<one sentence>"
}}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": judge_prompt}],
        temperature=0,
        max_tokens=300
    )
    
    result = json.loads(response.choices[0].message.content)
    return RAGEvaluation(**result)


#Run evaluation
results = []

for group in test_groups:
    reset_conversation()  # reset only between groups
    
    for item in group:
        # Use rag() so query rewriting kicks in automatically
        rag_answer = rag(item["question"], top_k=5, verbose=False)
        
        # Get context for the judge
        search_query = rewrite_query(item["question"], conversation_history)
        relevant_chunks = retrieve_relevant_chunks(search_query, top_k=5)
        context = "\n".join([r['chunk']['text'] for r in relevant_chunks])
        
        # Judge it
        eval_result = evaluate_rag_answer(
            item["question"], item["expected"], rag_answer, context
        )
        
        results.append({
            "question": item["question"],
            "rag_answer": rag_answer,
            "faithfulness": eval_result.faithfulness_score,
            "correctness": eval_result.correctness_score,
            "faithfulness_reasoning": eval_result.faithfulness_reasoning,
            "correctness_reasoning": eval_result.correctness_reasoning,
        })
        
        print(f"Q: {item['question'][:60]}...")
        print(f"   Faithfulness: {eval_result.faithfulness_score}/5 — {eval_result.faithfulness_reasoning}")
        print(f"   Correctness:  {eval_result.correctness_score}/5 — {eval_result.correctness_reasoning}")
        print()

#Summary
avg_faith = sum(r['faithfulness'] for r in results) / len(results)
avg_correct = sum(r['correctness'] for r in results) / len(results)

print("=" * 60)
print(f"📊 EVALUATION SUMMARY ({len(results)} questions)")
print("=" * 60)
print(f"   Avg Faithfulness: {avg_faith:.1f}/5")
print(f"   Avg Correctness:  {avg_correct:.1f}/5")
print()
print("Per-question breakdown:")
for r in results:
    print(f"  [{r['faithfulness']}/5 faith | {r['correctness']}/5 correct] {r['question'][:55]}...")

Conversation history cleared!
Q: What is the Yellow Ribbon Program at Fordham?...
   Faithfulness: 5/5 — The RAG system's answer accurately describes the Yellow Ribbon Program at Fordham and its relation to the Post-9/11 GI Bill, including details about tuition coverage and eligibility criteria.
   Correctness:  5/5 — The information provided in the RAG system's answer is factually correct and aligns with the expected answer regarding the program's purpose and benefits.

Q: Where is Fordham's Rose Hill campus?...
   Faithfulness: 5/5 — The RAG system's answer accurately reflects the location of Fordham's Rose Hill campus, providing the exact address and relevant contextual information.
   Correctness:  5/5 — The answer is correct as it includes the Bronx, New York, and additional accurate details about the campus.

Q: What is the Gabelli School of Business?...
   Faithfulness: 5/5 — The RAG system's answer accurately reflects the key aspects of the Gabelli School of Business, including

---

## Step 7: Build a Streamlit App

Your RAG system lives inside a notebook — that's great for development, but nobody is going to use a Jupyter notebook to ask questions about Fordham. Turn it into a web app using [Streamlit](https://docs.streamlit.io/).

Create a `.py` file (e.g., `scripts/fordham_rag_app.py`) that:
1. Lets the user type a question about Fordham
2. Runs your RAG pipeline
3. Displays the answer and the source pages used

**Getting started:**
- Install: `uv pip install streamlit`
- Run: `streamlit run scripts/fordham_rag_app.py`

**Tip**: Use `@st.cache_resource` to avoid reloading embeddings on every interaction.

**Include a screenshot of your working app below.**

![Fordham RAG App](app_screenshot.png)

---

## Step 8: How to Run Your System

Fill in the details below so that I can run and test your RAG system.

| Item | Your Answer |
|------|-------------|
| **Embedding model used** | `sentence-transformers/all-MiniLM-L6-v2` (local model) |
| **LLM used for generation** | `gpt-4o-mini` via OpenAI API |
| **LLM used for evaluation (judge)** | `gpt-4o-mini` via OpenAI API |
| **Saved artifacts** | `embeddings.npy` (chunk vectors), `chunks.json` (chunk text + metadata) |
| **How to start the Streamlit app** | `streamlit run fordham_rag_app.py` |
| **Any API keys or env vars needed** | (`OPENAI_API_KEY` in a `.env` file in the root folder |
| **Anything else I should know** | AI assistance: Claude (Anthropic) was used to help design and debug the RAG pipeline |

---

## Bonus: Experiment and Improve

Now that you have a working RAG system *and* a way to measure its quality, try to improve it. Use your evaluation framework to measure the impact of changes.

Ideas: different chunk sizes, different embedding models, hybrid search, better prompts, reranking, query rewriting. Document what you tried and show before/after evaluation scores.

In [ ]:
# BONUS: Experiment with larger chunk size
# Current system uses 1000 char chunks → avg 4.8/5 faithfulness, 4.8/5 correctness
# Hypothesis: larger chunks (1500 chars) give more context → better answers

# Step 1: Re-chunk with size 1500
print("Re-chunking with size 1500...")
bonus_chunks = []
chunk_id = 0

for i, doc in enumerate(documents):
    content = doc['content']
    
    if len(content) <= 1500:
        bonus_chunks.append({
            'chunk_id': chunk_id,
            'doc_filename': doc['filename'],
            'doc_url': doc['url'],
            'text': content
        })
        chunk_id += 1
    else:
        start = 0
        while start < len(content):
            end = start + 1500
            if end < len(content):
                last_space = content[start:end].rfind(' ')
                if last_space > 0:
                    end = start + last_space
            chunk = content[start:end].strip()
            if chunk:
                bonus_chunks.append({
                    'chunk_id': chunk_id,
                    'doc_filename': doc['filename'],
                    'doc_url': doc['url'],
                    'text': chunk
                })
                chunk_id += 1
            start = end

print(f"Done! Created {len(bonus_chunks)} chunks (vs 45,020 before)")

# Step 2: Re-embed the new chunks
print("\nEmbedding new chunks (this will take a few minutes)...")
bonus_chunk_texts = [c['text'] for c in bonus_chunks]
bonus_embeddings = model.encode(bonus_chunk_texts, show_progress_bar=True, batch_size=32)
print(f"Done! Embeddings shape: {bonus_embeddings.shape}")

Re-chunking with size 1500...
Done! Created 31544 chunks (vs 45,020 before)

Embedding new chunks (this will take a few minutes)...


Batches: 100%|██████████| 986/986 [02:50<00:00,  5.78it/s]

Done! Embeddings shape: (31544, 384)


In [ ]:
# Step 3: Evaluate the new system and compare scores

def retrieve_bonus(question, top_k=5):
    """Retrieve using the new larger-chunk embeddings."""
    q_vec = model.encode([question])
    sims = cosine_similarity(q_vec, bonus_embeddings)[0]
    top_indices = np.argsort(sims)[-top_k:][::-1]
    return [{"chunk": bonus_chunks[i], "similarity": float(sims[i])} for i in top_indices]


# Run same evaluation but with bonus retriever
bonus_results = []
bonus_history = []

for group in test_groups:
    bonus_history = []  # reset between groups
    
    for item in group:
        # Retrieve with new chunks
        relevant_chunks = retrieve_bonus(item["question"], top_k=5)
        context = "\n".join([r['chunk']['text'] for r in relevant_chunks])
        
        # Generate with history
        rag_answer = generate_answer(item["question"], relevant_chunks, history=bonus_history)
        
        bonus_history.append({"question": item["question"], "answer": rag_answer})
        if len(bonus_history) > 5:
            bonus_history.pop(0)
        
        eval_result = evaluate_rag_answer(
            item["question"], item["expected"], rag_answer, context
        )
        
        bonus_results.append({
            "question": item["question"],
            "faithfulness": eval_result.faithfulness_score,
            "correctness": eval_result.correctness_score,
        })

# Step 4: Compare before vs after
bonus_faith = sum(r['faithfulness'] for r in bonus_results) / len(bonus_results)
bonus_correct = sum(r['correctness'] for r in bonus_results) / len(bonus_results)

print("=" * 60)
print("📊 BEFORE vs AFTER COMPARISON")
print("=" * 60)
print(f"{'Metric':<20} {'Before (1000 chars)':<25} {'After (1500 chars)'}")
print("-" * 60)
print(f"{'Avg Faithfulness':<20} {'4.8/5':<25} {bonus_faith:.1f}/5")
print(f"{'Avg Correctness':<20} {'4.8/5':<25} {bonus_correct:.1f}/5")
print(f"{'Total chunks':<20} {'45,020':<25} {len(bonus_chunks)}")
print()
print("Per-question breakdown (new system):")
for r in bonus_results:
    print(f"  [{r['faithfulness']}/5 faith | {r['correctness']}/5 correct] {r['question'][:55]}...")

print()
print("=" * 60)
print(" CONCLUSION")
print("=" * 60)
print("""
Experiment: Does increasing chunk size (1000 → 1500 chars) improve RAG quality?

Result: NO — smaller chunks performed better.

  Before (1000 chars): Faithfulness 4.8/5 | Correctness 4.8/5
  After  (1500 chars): Faithfulness 4.6/5 | Correctness 4.5/5

Why smaller chunks won:
- Smaller chunks are more FOCUSED — each chunk is about one specific topic
- When retrieving top 5 chunks, smaller chunks give 5 different specific topics
- Larger chunks mix multiple topics together, making retrieval less precise
- The "How many campuses" question dropped from 4/5 to 1/5 with larger chunks,
  suggesting the campus info got buried inside a longer chunk about something else

Takeaway: For a university website with many short focused pages, 
1000 character chunks are the sweet spot. Larger chunks hurt precision.
""")

📊 BEFORE vs AFTER COMPARISON
Metric               Before (1000 chars)       After (1500 chars)
------------------------------------------------------------
Avg Faithfulness     4.8/5                     4.6/5
Avg Correctness      4.8/5                     4.6/5
Total chunks         45,020                    31544

Per-question breakdown (new system):
  [5/5 faith | 5/5 correct] What is the Yellow Ribbon Program at Fordham?...
  [5/5 faith | 5/5 correct] Where is Fordham's Rose Hill campus?...
  [5/5 faith | 5/5 correct] What is the Gabelli School of Business?...
  [4/5 faith | 4/5 correct] How do I apply for financial aid at Fordham?...
  [5/5 faith | 5/5 correct] What graduate programs does the School of Education off...
  [5/5 faith | 5/5 correct] What is the FAFSA?...
  [2/5 faith | 2/5 correct] How many campuses does Fordham have?...
  [5/5 faith | 5/5 correct] What is Fordham's Jesuit identity?...
  [5/5 faith | 5/5 correct] Does Fordham have a law school?...
  [5/5 faith | 5/5 co

---

## Git Submission

- [ ] Create a new branch called `homework-5`
- [ ] Commit your work (notebook + Streamlit app + saved artifacts in `temp/`)
- [ ] Push to GitHub
- [ ] Create a Pull Request and merge to main
- [ ] Submit the `.ipynb` file on Blackboard